# 04 · TCGA: second training site and held-out test, by race, sex and age

**Why.** UCSF-PDGM is one hospital and has no race field. Models trained on it alone lost accuracy on other hospitals' scanners. The TCGA-GBM/LGG cohort comes from many US hospitals and records race, so it is used twice:

- **70% of TCGA patients join training** (`ml/training/train_all_variants.py`), next to the 395 UCSF training patients.
- **30% are held out** and scored here by the deployed models, which never saw them.

The split is fixed per patient (`tcga_split`: stratified by IDH status and race, seed 0) and stored in the `split` column.

Sources: BraTS 2020 training set (pre-operative TCGA scans and expert outlines), cBioPortal `lgggbm_tcga_pub` (IDH, 1p/19q, MGMT, grade, age, sex), NCI Genomic Data Commons (race). Features: `ml/pipeline/extract.py` mode v1, as for UCSF. Only derived features are committed.

Race is **evaluated, never a model input**: it is a social category, not a measurement, and as an input it could encode existing bias.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd

REPO=Path.cwd().resolve().parents[1]
sys.path[:0]=[str(REPO/"backend"), str(REPO/"ml"/"pipeline")]
import external_validation as ev
from app.core.logging import configure_logging
configure_logging("WARNING")

table=pd.read_parquet(ev.FEATURES)
assert (table["split"]==ev.tcga_split(table)).all(), "stored split differs from tcga_split"
pd.crosstab(table["race"], table["split"], margins=True)

split,test,train,All
race,,,
Asian,2,1,3
Black or African American,3,9,12
Other or not reported,2,3,5
White,44,103,147
All,51,116,167


## Held-out patients

In [2]:
test=table[table["split"]=="test"]
pd.DataFrame({
    "patients": [len(test)], "female": [int((test["sex"]==0).sum())], "male": [int((test["sex"]==1).sum())],
    "median age": [test["age"].median()], "IDH-mutant": [int((test["idh"]==1).sum())],
    "1p/19q codeleted": [int((test["codeletion_1p19q"]==1).sum())], "grade 4": [int((test["who_grade"]==4).sum())],
})

,patients,female,male,median age,IDH-mutant,1p/19q codeleted,grade 4
0,51,23,28,57.0,18,4,31


## Deployed models on the held-out patients

In [3]:
preds=ev.predict(test)
result=ev.report(test, preds)
pd.DataFrame({m: e["overall"] for m, e in result["markers"].items()}).T

,n,positives,accuracy,auc,balanced_accuracy,stable_n,stable_accuracy
1p19q,43.0,4.0,0.9070,0.6859,0.5000,43.0,0.9070
GRADE,51.0,31.0,0.8627,0.9484,0.8250,47.0,0.8936
IDH,45.0,18.0,0.9111,0.9856,0.8889,41.0,0.9512
MGMT,33.0,24.0,0.7273,0.2778,0.5000,NaN,NaN


## By race, sex and age

AUC: how well patients are ranked (0.5 = chance). Accuracy: calls at the model's cut-off. With 3 Black and 2 Asian held-out patients, per-race numbers are not comparable.

In [4]:
def by(group):
    rows=[]
    for m, e in result["markers"].items():
        for g, x in e[f"by_{group}"].items():
            rows.append({"marker": m, group: g, **x})
    return pd.DataFrame(rows).set_index(["marker", group])
by("race")

n  positives  accuracy  stable_n  \
marker race                                                           
1p19q  Asian                       1          0    1.0000       1.0   
       Black or African American   3          0    1.0000       3.0   
       Other or not reported       1          0    1.0000       1.0   
       White                      38          4    0.8947      38.0   
GRADE  Asian                       2          2    1.0000       2.0   
       Black or African American   3          1    0.6667       2.0   
       Other or not reported       2          2    1.0000       2.0   
       White                      44         26    0.8636      41.0   
IDH    Asian                       1          0    1.0000       1.0   
       Black or African American   3          2    1.0000       3.0   
       Other or not reported       2          0    1.0000       2.0   
       White                      39         16    0.8974      35.0   
MGMT   Black or African American   3          3    1.0000       NaN   
       Other or not reported       1          0    0.0000       NaN   
       White                      29         21    0.7241       NaN   

                                  stable_accuracy     auc  balanced_accuracy  
marker race                                                                   
1p19q  Asian                               1.0000     NaN                NaN  
       Black or African American           1.0000     NaN                NaN  
       Other or not reported               1.0000     NaN                NaN  
       White                               0.8947  0.6838             0.5000  
GRADE  Asian                               1.0000     NaN                NaN  
       Black or African American           1.0000  1.0000             0.7500  
       Other or not reported               1.0000     NaN                NaN  
       White                               0.8780  0.9551             0.8333  
IDH    Asian                               1.0000     NaN                NaN  
       Black or African American           1.0000  1.0000             1.0000  
       Other or not reported               1.0000     NaN                NaN  
       White                               0.9429  0.9837             0.8750  
MGMT   Black or African American              NaN     NaN                NaN  
       Other or not reported                  NaN     NaN                NaN  
       White                                  NaN  0.2202             0.5000

In [5]:
by("sex")

n  positives  accuracy     auc  balanced_accuracy  stable_n  \
marker sex                                                                    
1p19q  female  21          3    0.8571  0.6111             0.5000      21.0   
       male    22          1    0.9545  0.8571             0.5000      22.0   
GRADE  female  23         11    0.7826  0.9091             0.7917      21.0   
       male    28         20    0.9286  0.9875             0.8750      26.0   
IDH    female  21         12    0.8095  0.9815             0.8333      17.0   
       male    24          6    1.0000  1.0000             1.0000      24.0   
MGMT   female  18         15    0.8333  0.3556             0.5000       NaN   
       male    15          9    0.6000  0.2222             0.5000       NaN   

               stable_accuracy  
marker sex                      
1p19q  female           0.8571  
       male             0.9545  
GRADE  female           0.8571  
       male             0.9231  
IDH    female           0.8824  
       male             1.0000  
MGMT   female              NaN  
       male                NaN

In [6]:
by("age_band")

n  positives  accuracy     auc  balanced_accuracy  stable_n  \
marker age_band                                                                 
1p19q  45-64     20          4    0.8000  0.7812             0.5000      20.0   
       65+       10          0    1.0000     NaN                NaN      10.0   
       under 45  13          0    1.0000     NaN                NaN      13.0   
GRADE  45-64     23         15    0.8261  0.9167             0.7500      22.0   
       65+       15         13    0.9333  0.9615             0.7500      14.0   
       under 45  13          3    0.8462  1.0000             0.9000      11.0   
IDH    45-64     20          7    0.8000  0.9780             0.7143      17.0   
       65+       12          1    1.0000  1.0000             1.0000      12.0   
       under 45  13         10    1.0000  1.0000             1.0000      12.0   
MGMT   45-64     14          9    0.6429  0.1111             0.5000       NaN   
       65+        8          5    0.6250  0.5333             0.5000       NaN   
       under 45  11         10    0.9091  0.6000             0.5000       NaN   

                 stable_accuracy  
marker age_band                   
1p19q  45-64              0.8000  
       65+                1.0000  
       under 45           1.0000  
GRADE  45-64              0.8636  
       65+                0.9286  
       under 45           0.9091  
IDH    45-64              0.8824  
       65+                1.0000  
       under 45           1.0000  
MGMT   45-64                 NaN  
       65+                   NaN  
       under 45              NaN

## Matches the committed report used by the app

In [7]:
committed=json.loads(ev.METRICS.read_text())
assert committed["markers"]==json.loads(json.dumps(result["markers"])), "recomputed metrics differ"
print("recomputed metrics match artifacts/metrics/external_tcga.json")

recomputed metrics match artifacts/metrics/external_tcga.json

## Reading the results

- IDH and grade transfer well to hospitals the models were not trained on. To score the previous models (trained on UCSF alone) on the same patients: `git checkout 53db667 -- artifacts/models artifacts/manifest.json`, then `backend/.venv/Scripts/python ml/pipeline/external_validation.py --score`; that run gave IDH 80% and grade 65% accuracy.
- 1p/19q has only a handful of co-deleted held-out patients and MGMT has no reliable MRI signal; neither is a usable call.
- TCGA is mostly White, so race diversity stays limited even with two sources; the app states this next to the table.